# LGCP hyperparameter and kernel tuning for the paper

Selects the LGCP's learning rate and temporal kernel structure. Baseline
hyperparameter selection is handled separately in
`51_paper_baseline_hyperparameter_selection.ipynb`; this notebook only
covers the LGCP.

Covariates are fixed (not searched): `chl, thetao, fishing_block,
is_holiday, is_weekend, cell_lag_1, cell_lag_7`. `num_steps=5000`,
`M_inducing=300`, `num_mc=128`, `minibatch=1024` are fixed at their
requested/recommended values for the **final confirmation run only**; the
two search stages below use a cheaper setting (`num_steps=1500,
num_mc=32`) purely to rank candidates fast, then retrain the winning
config at full budget.

**Method**

1. **Learning-rate search (Stage 1)**: with the current default temporal
   kernel (RBF + weekly periodic), evaluate lr &isin; {0.0001, 0.001,
   0.01} on 4 validation weeks. Take the top-2 learning rates and insert 2
   log-spaced intermediates between them; evaluate those too. Rank all 5
   by the relative, week-averaged score and keep the best.
2. **Kernel search (Stage 2)**: with the learning rate fixed at the Stage
   1 winner, evaluate 6 temporal kernel structures (spatial kernel stays
   RBF throughout, as requested): single-periodic (week / month / year)
   and multi-periodic (week+month, week+year, week+month+year), each on
   top of a short-scale RBF component. Rank the same way.
3. **Final confirmation**: retrain the winning (lr, kernel) config at full
   budget (`num_steps=5000`, `num_mc=128`) on the paper's reporting weeks
   (May and November 2025 -- the same weeks used in
   `scripts/17_run_best_baselines.py`, deliberately disjoint from the 4
   validation weeks above), then slice predictions into the same 3
   forecast horizons (full week / first 3 days / first day) used for the
   baseline tables, producing 3 CSVs with just the LGCP row.

The zero-inflation gate (`src/models/zero_gate.py`) is **disabled**
throughout this notebook: it was compared against plain LGCP in
`52_zero_gate_lgcp_comparison.ipynb` and found to hurt performance, so
the fairest LGCP row to report is the plain model.

**Note on temporal periods**: the current production config
(`config/train_basic_lgcp_multi_kernel.yaml`) has its weekly periodic
component set to `period: 0.074`, labeled "is a week" -- this is stale.
With the corrected multi-year `t_norm` (continuous over the full
2024-01-01..2025-12-31 span, 730 days), a real week is `7/730 &approx;
0.0096` in normalized units, not 0.074. This notebook recomputes the
week/month/year periods directly from the data span below and uses the
corrected values throughout.

## Grids and fixed settings

In [1]:
YEARS = [2024, 2025]
SEED = 0

# Validation weeks for both search stages -- same 4 weeks used for baseline
# hyperparameter selection in notebook 51, deliberately disjoint from the
# May/November reporting weeks used for the final confirmation below.
VALIDATION_WEEKS = [
    {"label": "april_w1", "test_start": "2025-04-17", "test_end": "2025-04-23"},
    {"label": "april_w2", "test_start": "2025-04-24", "test_end": "2025-04-30"},
    {"label": "october_w1", "test_start": "2025-10-18", "test_end": "2025-10-24"},
    {"label": "october_w2", "test_start": "2025-10-25", "test_end": "2025-10-31"},
]

# Reporting weeks for the final confirmation -- same as scripts/17_run_best_baselines.py
REPORT_WEEKS = [
    {"label": "may", "test_start": "2025-05-01", "test_end": "2025-05-07"},
    {"label": "november", "test_start": "2025-11-01", "test_end": "2025-11-07"},
]

HORIZONS = [
    ("full_week", 7),
    ("first_3_days", 3),
    ("first_day", 1),
]

METRIC_KEYS = [
    "mean_ll_obs", "mae_obs", "rmse_obs",
    "mean_ll_daily", "mae_daily", "rmse_daily",
    "daily_delta_corr", "daily_direction_accuracy_moving",
]

METRIC_DIRECTIONS = {
    "mean_ll_obs": "max",
    "rmse_daily": "min",
}

FIXED_COVARIATE_COLS = [
    "chl", "thetao", "fishing_block", "is_holiday", "is_weekend",
    "cell_lag_1", "cell_lag_7",
]

# --- search-phase settings (cheap, for ranking candidates) ---
SEARCH_NUM_STEPS = 1500
SEARCH_NUM_MC = 32

# --- final settings (as requested) ---
FINAL_NUM_STEPS = 5000
M_INDUCING = 300
FINAL_NUM_MC = 128
MINIBATCH = 1024  # kept at 1024: an empirical timing probe found minibatch=512
                   # saved almost no time (the fixed M=300 inducing-point Cholesky
                   # dominates the per-step cost far more than the minibatch-linear
                   # term), so there's no real efficiency case for shrinking it,
                   # and a larger minibatch gives a less noisy gradient estimate.

LR_GRID_INITIAL = [1e-4, 1e-3, 1e-2]

BASE_LGCP_CONFIG = "config/train_basic_lgcp_multi_kernel.yaml"
RESULTS_DIR_NAME = "reports/paper/lgcp_two_week_2025_05_11"  # same dir as script 17's baseline CSVs

print(f"Validation weeks: {len(VALIDATION_WEEKS)}   Report weeks: {len(REPORT_WEEKS)}")
print(f"Search settings:  num_steps={SEARCH_NUM_STEPS}  num_mc={SEARCH_NUM_MC}  minibatch={MINIBATCH}")
print(f"Final settings:   num_steps={FINAL_NUM_STEPS}  num_mc={FINAL_NUM_MC}  M_inducing={M_INDUCING}  minibatch={MINIBATCH}")

Validation weeks: 4   Report weeks: 2
Search settings:  num_steps=1500  num_mc=32  minibatch=1024
Final settings:   num_steps=5000  num_mc=128  M_inducing=300  minibatch=1024


## Setup: imports, module loading, corrected periods

In [2]:
from __future__ import annotations

import importlib.util
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.metrics_lgcp import evaluate_metrics
from src.data.multi_year import load_parquet_years


def load_script_module(name, relpath):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / relpath)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


train_lgcp = load_script_module("train_lgcp", "scripts/11_train_lgcp.py")

RESULTS_DIR = PROJECT_ROOT / RESULTS_DIR_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- corrected periods, computed directly from the actual multi-year data span ---
_df_span = load_parquet_years(str(PROJECT_ROOT / "data/processed/cpr_gfw.parquet"), YEARS)
_df_span["date"] = pd.to_datetime(_df_span["date"])
_t_min, _t_max = _df_span["date"].min(), _df_span["date"].max()
_span_days = (_t_max - _t_min).days

PERIODS = {
    "week": 7 / _span_days,
    "month": 30.44 / _span_days,
    "year": 365.25 / _span_days,
}

print(f"Data span: {_t_min.date()} .. {_t_max.date()}  ({_span_days} days)")
print(f"Corrected periods (t_norm units): {PERIODS}")
print(f"Modules loaded OK. Results will be saved under: {RESULTS_DIR}")

Data span: 2024-01-01 .. 2025-12-31  (730 days)
Corrected periods (t_norm units): {'week': 0.009589041095890411, 'month': 0.041698630136986305, 'year': 0.5003424657534247}
Modules loaded OK. Results will be saved under: /home/rdoz/PhD/AIPS/reports/paper/lgcp_two_week_2025_05_11


## Methodology helper

**`compute_relative_scores`**: for each metric, min-max normalizes across
the hyperparameter combos *within each week* (1.0 = best that week), then
averages across weeks per combo. A combo is only kept if it converged
(produced valid results) on every week. `combined_relative_score` is the
mean of all relative metric columns and is what every search in this
notebook ranks by. (Identical helper to notebook 51.)

In [3]:
def compute_relative_scores(df: pd.DataFrame, week_col: str, group_cols: list[str], metrics_directions: dict) -> pd.DataFrame:
    group_cols = list(group_cols)
    valid_df = df.dropna(subset=list(metrics_directions)).copy()

    for metric, direction in metrics_directions.items():
        def _normalize(s, direction=direction):
            lo, hi = s.min(), s.max()
            if hi - lo < 1e-12:
                return pd.Series(1.0, index=s.index)
            return (s - lo) / (hi - lo) if direction == "max" else (hi - s) / (hi - lo)
        valid_df[f"relative_{metric}"] = valid_df.groupby(week_col)[metric].transform(_normalize)

    relative_cols = [f"relative_{m}" for m in metrics_directions]
    n_weeks_total = df[week_col].nunique()
    week_counts = valid_df.groupby(group_cols)[week_col].nunique().rename("n_weeks_valid")

    scores = valid_df.groupby(group_cols)[relative_cols].mean()
    scores["combined_relative_score"] = scores[relative_cols].mean(axis=1)
    scores = scores.join(week_counts).reset_index()

    complete = scores[scores["n_weeks_valid"] == n_weeks_total].copy()
    incomplete = scores[scores["n_weeks_valid"] != n_weeks_total]
    if len(incomplete):
        print(f"  Excluded {len(incomplete)} combo(s) that didn't converge on every week:")
        display(incomplete[group_cols + ["n_weeks_valid"]])
    return complete.sort_values("combined_relative_score", ascending=False)


print("Helper defined.")

Helper defined.


## Kernel builder

Spatial kernel is fixed (single RBF, matching the validated production
default: `lengthscale=0.1` fixed, `variance` trainable). The temporal
kernel is always a short-scale RBF (`lengthscale=0.01` fixed, `variance`
trainable) plus one or more periodic components at the corrected
week/month/year periods (`lengthscale=0.2`, `variance=0.2`, both
trainable; `period` fixed/non-trainable since it is a known physical
constant). `temporal_composition="sum"` (additive components) and
`spatial_composition="sum"` (moot here, single component); the
space-time combination itself defaults to `product` (separable), which
was confirmed better than `sum` in earlier experiments.

In [4]:
def _spatial_kernels():
    return [{
        "type": "RBF",
        "hyperparameters": {"lengthscale": 0.1, "variance": 2.0},
        "trainable": {"lengthscale": False, "variance": True},
    }]


def _rbf_short_scale():
    return {
        "type": "RBF",
        "hyperparameters": {"lengthscale": 0.01, "variance": 0.5},
        "trainable": {"lengthscale": False, "variance": True},
    }


def _periodic(period_name):
    return {
        "type": "periodic",
        "hyperparameters": {"lengthscale": 0.2, "variance": 0.2, "period": PERIODS[period_name]},
        "trainable": {"lengthscale": True, "variance": True, "period": False},
    }


# variant name -> list of period names combined with the temporal RBF
TEMPORAL_KERNEL_VARIANTS = {
    "week": ["week"],
    "month": ["month"],
    "year": ["year"],
    "week_month": ["week", "month"],
    "week_year": ["week", "year"],
    "week_month_year": ["week", "month", "year"],
}

DEFAULT_VARIANT = "week"  # used for the Stage 1 learning-rate search


def build_model_cfg(variant_name: str) -> dict:
    temporal_kernels = [_rbf_short_scale()] + [_periodic(p) for p in TEMPORAL_KERNEL_VARIANTS[variant_name]]
    return {
        "spatial_composition": "sum",
        "temporal_composition": "sum",
        "spatial_kernels": _spatial_kernels(),
        "temporal_kernels": temporal_kernels,
    }


print(f"Kernel variants: {list(TEMPORAL_KERNEL_VARIANTS)}")
print(f"Default variant for LR search: {DEFAULT_VARIANT}")

Kernel variants: ['week', 'month', 'year', 'week_month', 'week_year', 'week_month_year']
Default variant for LR search: week


## LGCP runner

Trains one `SparseLGCP` and evaluates it on the full test week (no
horizon slicing during search -- that's only needed for the final
report). Zero-gate stays disabled throughout.

Some (lr, kernel, week) combinations can make the inducing-point
covariance `Kuu` numerically singular partway through training (a
pre-existing numerical-stability edge case in `SparseLGCP.elbo_mc`,
independent of this notebook), raising a Cholesky `LinAlgError`. Rather
than patch the shared model code, `run_lgcp` retries once with a
different inducing-point seed (which resamples `Z` and usually avoids the
pathological configuration) and, if that also fails, returns `None` --
the caller skips that single (candidate, week) row, and
`compute_relative_scores` already drops any candidate that doesn't have a
valid result on every week.

In [5]:
def _run_lgcp_once(week: dict, lr: float, model_cfg: dict, num_steps: int, num_mc: int, np_seed: int):
    label, test_start, test_end = week["label"], week["test_start"], week["test_end"]

    base_cfg = train_lgcp.load_config(str(PROJECT_ROOT / BASE_LGCP_CONFIG))
    base_cfg["years"] = YEARS
    base_cfg["parquet_path"] = str(PROJECT_ROOT / base_cfg["parquet_path"])
    base_cfg["covariate_cols"] = FIXED_COVARIATE_COLS
    base_cfg["M_inducing"] = M_INDUCING
    base_cfg["model"] = model_cfg

    (train_coords, train_covs, train_y, test_coords, test_covs, test_y, scalers, df) = train_lgcp.prepare_data(
        base_cfg["parquet_path"], train_fraction=base_cfg["train_fraction"], random_seed=base_cfg["data_seed"],
        split_strategy="fixed_test_window", covariate_cols=base_cfg["covariate_cols"],
        test_start_date=test_start, test_end_date=test_end,
        lag_features=base_cfg.get("lag_features"), years=YEARS,
    )
    df["date"] = pd.to_datetime(df["date"])
    _, test_mask = train_lgcp.make_day_split_masks(
        df=df, train_fraction=base_cfg["train_fraction"], random_seed=base_cfg["data_seed"],
        split_strategy="fixed_test_window", test_start_date=test_start, test_end_date=test_end,
    )
    test_dates = df.loc[test_mask, "date"].values

    run_cfg = dict(base_cfg)
    run_cfg["kernel_config"] = train_lgcp.normalize_kernel_config(run_cfg)
    run_cfg["lr"] = lr
    run_cfg["num_mc"] = num_mc
    run_cfg["num_steps"] = num_steps
    run_cfg["minibatch"] = MINIBATCH
    run_cfg["log_every"] = max(num_steps, 1)
    run_cfg["np_seed"] = np_seed

    train_lgcp.set_seeds(np_seed)
    model = train_lgcp.SparseLGCP(
        train_coords, train_covs, train_y, kernel_config=run_cfg["kernel_config"],
        M_inducing=run_cfg["M_inducing"], device=run_cfg["device"], np_seed=np_seed,
    )
    train_lgcp.configure_log_noise(model, run_cfg)
    train_lgcp.train(model, run_cfg)

    rate_mean_test, _, _ = model.predict_rate(test_coords, test_covs, num_samples=run_cfg["num_pred_samples"])
    metrics = evaluate_metrics(test_y, rate_mean_test, test_dates)
    return {
        "week": label, "test_start": test_start,
        **{k: metrics[k] for k in METRIC_KEYS},
    }, (test_dates, test_y, rate_mean_test)


def run_lgcp(week: dict, lr: float, model_cfg: dict, num_steps: int, num_mc: int, np_seed: int = 0, raise_on_failure: bool = False):
    try:
        return _run_lgcp_once(week, lr, model_cfg, num_steps, num_mc, np_seed)
    except torch.linalg.LinAlgError as e:
        print(f"  [WARN] Kuu Cholesky failed (lr={lr}, week={week['label']}, seed={np_seed}): {e}. Retrying with a different inducing-point seed …")
        try:
            return _run_lgcp_once(week, lr, model_cfg, num_steps, num_mc, np_seed + 997)
        except torch.linalg.LinAlgError as e2:
            if raise_on_failure:
                raise
            print(f"  [WARN] Retry also failed (lr={lr}, week={week['label']}): {e2}. Skipping this (candidate, week).")
            return None


print("Runner defined.")

Runner defined.


## Stage 1a: initial learning-rate search

lr &isin; {0.0001, 0.001, 0.01}, default kernel (`week`), all 4 validation
weeks, search-phase settings.

In [6]:
lr_search_rows = []
t0 = time.time()
for lr in LR_GRID_INITIAL:
    model_cfg = build_model_cfg(DEFAULT_VARIANT)
    for week in VALIDATION_WEEKS:
        result = run_lgcp(week, lr, model_cfg, SEARCH_NUM_STEPS, SEARCH_NUM_MC)
        if result is None:
            continue
        row, _ = result
        row["lr"] = lr
        lr_search_rows.append(row)
        print(f"[Stage 1a | lr={lr} | {week['label']}] mean_ll_obs={row['mean_ll_obs']:.4f}  rmse_daily={row['rmse_daily']:.4f}  ({time.time()-t0:.0f}s elapsed)")

lr_search_df = pd.DataFrame(lr_search_rows)
print(f"\nStage 1a done: {len(lr_search_df)} runs in {time.time()-t0:.0f}s")

Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False


[     1/1500]  loss=16596.1562  ll=-732.0956  kl=61.0911  (0.0s)


[  1500/1500]  loss=13395.0576  ll=-591.6016  kl=33.1811  (52.4s)
[Stage 1a | lr=0.0001 | april_w1] mean_ll_obs=-0.5400  rmse_daily=9.9276  (53s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=18368.6699  ll=-798.7331  kl=60.9899  (0.0s)


[  1500/1500]  loss=13261.1201  ll=-577.1247  kl=32.9032  (51.3s)
[Stage 1a | lr=0.0001 | april_w2] mean_ll_obs=-0.8806  rmse_daily=10.0546  (105s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=25613.9082  ll=-814.0276  kl=61.0746  (0.0s)


[  1500/1500]  loss=19652.2598  ll=-624.9128  kl=35.8555  (52.5s)
[Stage 1a | lr=0.0001 | october_w1] mean_ll_obs=-0.7004  rmse_daily=11.0990  (157s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=21258.4629  ll=-668.1483  kl=61.0681  (0.0s)


...

[  1500/1500]  loss=14861.3320  ll=-464.2246  kl=133.5333  (51.9s)
[Stage 1a | lr=0.001 | october_w2] mean_ll_obs=-0.5735  rmse_daily=6.4472  (419s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=16596.1562  ll=-732.0956  kl=61.0911  (0.0s)


[  1500/1500]  loss=9253.8691  ll=-405.3145  kl=99.4620  (50.4s)
[Stage 1a | lr=0.01 | april_w1] mean_ll_obs=-0.3617  rmse_daily=7.0798  (470s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=18368.6699  ll=-798.7331  kl=60.9899  (0.0s)


[  1500/1500]  loss=9191.4043  ll=-396.9190  kl=93.6627  (51.3s)
[Stage 1a | lr=0.01 | april_w2] mean_ll_obs=-0.6800  rmse_daily=11.9418  (521s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=25613.9082  ll=-814.0276  kl=61.0746  (0.0s)


[  1500/1500]  loss=13202.6514  ll=-417.0765  kl=110.3592  (50.6s)
[Stage 1a | lr=0.01 | october_w1] mean_ll_obs=-0.4562  rmse_daily=9.9910  (572s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=21258.4629  ll=-668.1483  kl=61.0681  (0.0s)


[  1500/1500]  loss=14125.7168  ll=-441.9562  kl=104.3973  (51.3s)
[Stage 1a | lr=0.01 | october_w2] mean_ll_obs=-0.5819  rmse_daily=6.5975  (624s elapsed)

Stage 1a done: 12 runs in 624s


## Stage 1a selection: top-2 learning rates and their intermediates

In [7]:
lr_scores_stage1a = compute_relative_scores(lr_search_df, week_col="week", group_cols=["lr"], metrics_directions=METRIC_DIRECTIONS)
print("Stage 1a ranking:")
display(lr_scores_stage1a)

top2 = lr_scores_stage1a.head(2)["lr"].tolist()
lr_lo, lr_hi = sorted(top2)
LR_INTERMEDIATES = np.geomspace(lr_lo, lr_hi, 4)[1:3].tolist()
print(f"\nTop-2 learning rates: {top2}")
print(f"2 log-spaced intermediates between them: {LR_INTERMEDIATES}")

Stage 1a ranking:


,lr,relative_mean_ll_obs,relative_rmse_daily,combined_relative_score,n_weeks_valid
2,0.0100,0.981996,0.867929,0.924963,4
1,0.0010,0.810649,0.701731,0.756190,4
0,0.0001,0.000000,0.250000,0.125000,4



Top-2 learning rates: [0.01, 0.001]
2 log-spaced intermediates between them: [0.0021544346900318843, 0.004641588833612777]


## Stage 1b: refined intermediate learning rates

In [8]:
for lr in LR_INTERMEDIATES:
    model_cfg = build_model_cfg(DEFAULT_VARIANT)
    for week in VALIDATION_WEEKS:
        result = run_lgcp(week, lr, model_cfg, SEARCH_NUM_STEPS, SEARCH_NUM_MC)
        if result is None:
            continue
        row, _ = result
        row["lr"] = lr
        lr_search_rows.append(row)
        print(f"[Stage 1b | lr={lr:.6f} | {week['label']}] mean_ll_obs={row['mean_ll_obs']:.4f}  rmse_daily={row['rmse_daily']:.4f}")

lr_search_df = pd.DataFrame(lr_search_rows)
print(f"\nStage 1 total: {len(lr_search_df)} runs across {lr_search_df['lr'].nunique()} distinct learning rates")

Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=16596.1562  ll=-732.0956  kl=61.0911  (0.0s)


[  1500/1500]  loss=9392.5703  ll=-409.5780  kl=141.8667  (51.0s)
[Stage 1b | lr=0.002154 | april_w1] mean_ll_obs=-0.3555  rmse_daily=6.7830
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=18368.6699  ll=-798.7331  kl=60.9899  (0.0s)


[  1500/1500]  loss=9228.7539  ll=-396.5070  kl=140.4566  (51.7s)
[Stage 1b | lr=0.002154 | april_w2] mean_ll_obs=-0.6924  rmse_daily=12.3433
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=25613.9082  ll=-814.0276  kl=61.0746  (0.0s)


  [WARN] Kuu Cholesky failed (lr=0.0021544346900318843, week=october_w1, seed=0): linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 1 is not positive-definite).. Retrying with a different inducing-point seed …
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=24583.0996  ll=-781.1908  kl=61.0332  (0.0s)


[  1500/1500]  loss=14274.1738  ll=-449.5797  kl=161.5845  (51.3s)
[Stage 1b | lr=0.002154 | october_w1] mean_ll_obs=-0.4617  rmse_daily=10.1883
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=21258.4629  ll=-668.1483  kl=61.0681  (0.0s)


...

  [WARN] Kuu Cholesky failed (lr=0.004641588833612777, week=april_w2, seed=0): linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 246 is not positive-definite).. Retrying with a different inducing-point seed …
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=17669.5898  ll=-768.2277  kl=61.1213  (0.0s)


[  1500/1500]  loss=9109.6006  ll=-392.6738  kl=109.1641  (50.5s)
[Stage 1b | lr=0.004642 | april_w2] mean_ll_obs=-0.7105  rmse_daily=15.7143
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=25613.9082  ll=-814.0276  kl=61.0746  (0.0s)


[  1500/1500]  loss=13238.2373  ll=-417.5874  kl=129.9070  (51.4s)
[Stage 1b | lr=0.004642 | october_w1] mean_ll_obs=-0.4546  rmse_daily=10.0725


Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=21258.4629  ll=-668.1483  kl=61.0681  (0.0s)


[  1500/1500]  loss=14110.1855  ll=-440.8060  kl=125.3555  (50.5s)
[Stage 1b | lr=0.004642 | october_w2] mean_ll_obs=-0.5953  rmse_daily=6.4978

Stage 1 total: 20 runs across 5 distinct learning rates


## Stage 1 final selection

In [9]:
lr_scores_final = compute_relative_scores(lr_search_df, week_col="week", group_cols=["lr"], metrics_directions=METRIC_DIRECTIONS)
print("Stage 1 final ranking (all 5 learning rates):")
display(lr_scores_final)

BEST_LR = float(lr_scores_final.iloc[0]["lr"])
print(f"\nBEST_LR = {BEST_LR}")

Stage 1 final ranking (all 5 learning rates):


,lr,relative_mean_ll_obs,relative_rmse_daily,combined_relative_score,n_weeks_valid
4,0.010000,0.971357,0.861579,0.916468,4
2,0.002154,0.942064,0.849826,0.895945,4
3,0.004642,0.915037,0.729604,0.822320,4
1,0.001000,0.801276,0.694816,0.748046,4
0,0.000100,0.000000,0.250000,0.125000,4



BEST_LR = 0.01


## Stage 2: temporal kernel structure search

Learning rate fixed at `BEST_LR`. Spatial kernel stays RBF. Reuses Stage
1's results for the `week` variant if `BEST_LR` was already tested there
(no need to retrain the identical config).

In [10]:
kernel_search_rows = []
t0 = time.time()
already_tested_lrs = set(lr_search_df["lr"].round(10))

for variant_name in TEMPORAL_KERNEL_VARIANTS:
    if variant_name == DEFAULT_VARIANT and round(BEST_LR, 10) in already_tested_lrs:
        reused = lr_search_df[np.isclose(lr_search_df["lr"], BEST_LR)].copy()
        for _, r in reused.iterrows():
            row = {k: r[k] for k in ["week", "test_start"] + METRIC_KEYS}
            row["kernel_variant"] = variant_name
            kernel_search_rows.append(row)
        print(f"[Stage 2 | variant={variant_name}] reused {len(reused)} Stage-1 result(s) at lr={BEST_LR}")
        continue

    model_cfg = build_model_cfg(variant_name)
    for week in VALIDATION_WEEKS:
        result = run_lgcp(week, BEST_LR, model_cfg, SEARCH_NUM_STEPS, SEARCH_NUM_MC)
        if result is None:
            continue
        row, _ = result
        row["kernel_variant"] = variant_name
        kernel_search_rows.append(row)
        print(f"[Stage 2 | variant={variant_name} | {week['label']}] mean_ll_obs={row['mean_ll_obs']:.4f}  rmse_daily={row['rmse_daily']:.4f}  ({time.time()-t0:.0f}s elapsed)")

kernel_search_df = pd.DataFrame(kernel_search_rows)
print(f"\nStage 2 done: {len(kernel_search_df)} rows across {kernel_search_df['kernel_variant'].nunique()} kernel variants in {time.time()-t0:.0f}s")

[Stage 2 | variant=week] reused 4 Stage-1 result(s) at lr=0.01
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=16588.9004  ll=-731.7730  kl=61.1211  (0.0s)


[  1500/1500]  loss=9249.4121  ll=-405.2984  kl=95.3678  (51.3s)
[Stage 2 | variant=month | april_w1] mean_ll_obs=-0.3637  rmse_daily=7.0411  (51s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=18386.7031  ll=-799.5165  kl=61.0673  (0.0s)


[  1500/1500]  loss=9155.0811  ll=-395.2442  kl=95.7283  (51.1s)
[Stage 2 | variant=month | april_w2] mean_ll_obs=-0.6762  rmse_daily=11.8785  (103s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=25651.3984  ll=-815.2203  kl=61.1242  (0.0s)


[  1500/1500]  loss=13164.6533  ll=-415.7893  kl=112.7669  (51.3s)
[Stage 2 | variant=month | october_w1] mean_ll_obs=-0.4528  rmse_daily=9.9477  (154s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=21257.5137  ll=-668.1163  kl=61.1312  (0.0s)


[  1500/1500]  loss=14029.5928  ll=-438.7702  kl=109.3509  (51.2s)
[Stage 2 | variant=month | october_w2] mean_ll_obs=-0.5934  rmse_daily=6.6271  (206s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=16622.4180  ll=-733.2441  kl=61.4130  (0.0s)


...

[  1500/1500]  loss=14052.2314  ll=-439.4921  kl=109.0869  (57.3s)
[Stage 2 | variant=week_year | october_w2] mean_ll_obs=-0.5972  rmse_daily=6.5797  (870s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=16659.8711  ll=-732.8014  kl=108.8657  (0.0s)


[  1500/1500]  loss=9212.7158  ll=-403.3379  kl=102.9508  (62.6s)
[Stage 2 | variant=week_month_year | april_w1] mean_ll_obs=-0.3680  rmse_daily=6.9192  (932s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=18453.5840  ll=-800.3572  kl=108.6770  (0.0s)


[  1500/1500]  loss=9153.7861  ll=-395.1334  kl=96.9735  (62.9s)
[Stage 2 | variant=week_month_year | april_w2] mean_ll_obs=-0.6773  rmse_daily=11.8243  (995s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=25686.1797  ll=-814.8111  kl=108.7492  (0.0s)


[  1500/1500]  loss=13230.2549  ll=-417.8539  kl=113.5596  (63.1s)
[Stage 2 | variant=week_month_year | october_w1] mean_ll_obs=-0.4558  rmse_daily=10.0156  (1059s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False
[     1/1500]  loss=21316.2715  ll=-668.4704  kl=108.6568  (0.0s)


[  1500/1500]  loss=14029.5459  ll=-438.5723  kl=115.5834  (62.7s)
[Stage 2 | variant=week_month_year | october_w2] mean_ll_obs=-0.5976  rmse_daily=6.6789  (1121s elapsed)

Stage 2 done: 24 rows across 6 kernel variants in 1121s


## Stage 2 selection

In [11]:
kernel_scores = compute_relative_scores(kernel_search_df, week_col="week", group_cols=["kernel_variant"], metrics_directions=METRIC_DIRECTIONS)
print("Stage 2 ranking:")
display(kernel_scores)

BEST_VARIANT = str(kernel_scores.iloc[0]["kernel_variant"])
print(f"\nBEST_VARIANT = {BEST_VARIANT}")
print(f"Winning config: lr={BEST_LR}, temporal_kernel={BEST_VARIANT} ({TEMPORAL_KERNEL_VARIANTS[BEST_VARIANT]})")

Stage 2 ranking:


,kernel_variant,relative_mean_ll_obs,relative_rmse_daily,combined_relative_score,n_weeks_valid
2,week_month,0.725180,0.636877,0.681029,4
0,month,0.721431,0.569046,0.645239,4
1,week,0.611698,0.398218,0.504958,4
4,week_year,0.306217,0.676954,0.491586,4
3,week_month_year,0.401922,0.557730,0.479826,4
5,year,0.434137,0.500000,0.467068,4



BEST_VARIANT = week_month
Winning config: lr=0.01, temporal_kernel=week_month (['week', 'month'])


## Final confirmation

Retrains the winning (lr, kernel) config at full budget
(`num_steps=5000`, `num_mc=128`, `M_inducing=300`, `minibatch=1024`) on
the May and November reporting weeks, then slices predictions into the
same 3 forecast horizons used for the baseline tables in
`scripts/17_run_best_baselines.py` (predictions computed once per week,
sliced by date cutoff -- no retraining per horizon).

In [12]:
def horizon_rows(method: str, week_label: str, test_dates, y_true, preds, test_start: str) -> list[dict]:
    test_dates = pd.to_datetime(np.asarray(test_dates))
    y_true = np.asarray(y_true)
    preds = np.asarray(preds)

    rows = []
    for horizon_label, horizon_days in HORIZONS:
        cutoff = pd.Timestamp(test_start) + pd.Timedelta(days=horizon_days - 1)
        mask = test_dates <= cutoff
        metrics = evaluate_metrics(y_true[mask], preds[mask], test_dates[mask])
        rows.append({
            "method": method,
            "week": week_label,
            "horizon": horizon_label,
            "horizon_days": horizon_days,
            **{k: metrics[k] for k in METRIC_KEYS},
        })
    return rows


FINAL_MODEL_CFG = build_model_cfg(BEST_VARIANT)

final_rows = []
t0 = time.time()
for week in REPORT_WEEKS:
    label = week["label"]
    _, (test_dates, test_y, preds) = run_lgcp(week, BEST_LR, FINAL_MODEL_CFG, FINAL_NUM_STEPS, FINAL_NUM_MC, np_seed=SEED, raise_on_failure=True)
    week_rows = horizon_rows("LGCP (ours)", label, test_dates, test_y, preds, week["test_start"])
    for r in week_rows:
        print(f"[Final | {label} | {r['horizon']}] mean_ll_obs={r['mean_ll_obs']:.4f}  rmse_daily={r['rmse_daily']:.4f}")
    final_rows += week_rows
    print(f"  ({time.time()-t0:.0f}s elapsed)")

final_df = pd.DataFrame(final_rows)
print(f"\nFinal confirmation done in {time.time()-t0:.0f}s")

Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False


[     1/5000]  loss=18461.5176  ll=-790.1298  kl=86.3715  (0.1s)


[  5000/5000]  loss=9865.5225  ll=-420.3402  kl=90.1492  (326.2s)
[Final | may | full_week] mean_ll_obs=-0.4139  rmse_daily=10.2946
[Final | may | first_3_days] mean_ll_obs=-0.2753  rmse_daily=1.9143
[Final | may | first_day] mean_ll_obs=-0.1099  rmse_daily=0.0387
  (326s elapsed)
Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']

=== Kernel noise configuration ===
noise_var: 0.00100000
trainable: False


[     1/5000]  loss=21502.1055  ll=-667.9659  kl=86.7523  (0.1s)


[  5000/5000]  loss=12964.4365  ll=-401.2320  kl=100.7207  (328.8s)
[Final | november | full_week] mean_ll_obs=-0.5492  rmse_daily=4.9059
[Final | november | first_3_days] mean_ll_obs=-0.2613  rmse_daily=2.8900
[Final | november | first_day] mean_ll_obs=-0.0052  rmse_daily=0.2555
  (655s elapsed)

Final confirmation done in 655s


## Save the 3 result tables

Same schema and output directory as `scripts/17_run_best_baselines.py`
(`best_baselines_metrics_{horizon}.csv`), but named `lgcp_metrics_{horizon}.csv`
so as not to overwrite the baseline files -- containing just the LGCP row,
ready to merge into the paper's existing tables.

In [13]:
for horizon_label, _ in HORIZONS:
    horizon_df = final_df[final_df["horizon"] == horizon_label].drop(columns=["horizon", "horizon_days"])
    out_path = RESULTS_DIR / f"lgcp_metrics_{horizon_label}.csv"
    horizon_df.to_csv(out_path, index=False)
    print(f"\n[{horizon_label}] saved -> {out_path}")
    display(horizon_df.set_index(["method", "week"]))

print(f"\nWinning config: lr={BEST_LR}, temporal_kernel={BEST_VARIANT} ({TEMPORAL_KERNEL_VARIANTS[BEST_VARIANT]})")
print(f"All 3 LGCP tables saved under {RESULTS_DIR}")


[full_week] saved -> /home/rdoz/PhD/AIPS/reports/paper/lgcp_two_week_2025_05_11/lgcp_metrics_full_week.csv


mean_ll_obs   mae_obs  rmse_obs  mean_ll_daily  \
method      week                                                       
LGCP (ours) may         -0.413883  0.257082  0.751226      -5.695202   
            november    -0.549231  0.311065  0.684812      -2.464516   

                      mae_daily  rmse_daily  daily_delta_corr  \
method      week                                                
LGCP (ours) may        6.902153   10.294555          0.301980   
            november   3.930856    4.905850          0.972104   

                      daily_direction_accuracy_moving  
method      week                                       
LGCP (ours) may                                  0.75  
            november                             1.00


[first_3_days] saved -> /home/rdoz/PhD/AIPS/reports/paper/lgcp_two_week_2025_05_11/lgcp_metrics_first_3_days.csv


mean_ll_obs   mae_obs  rmse_obs  mean_ll_daily  \
method      week                                                       
LGCP (ours) may         -0.275310  0.159202  0.384533      -1.972365   
            november    -0.261259  0.139029  0.426741      -1.226000   

                      mae_daily  rmse_daily  daily_delta_corr  \
method      week                                                
LGCP (ours) may        1.568828    1.914329               NaN   
            november   1.825420    2.889955               NaN   

                      daily_direction_accuracy_moving  
method      week                                       
LGCP (ours) may                                   1.0  
            november                              1.0


[first_day] saved -> /home/rdoz/PhD/AIPS/reports/paper/lgcp_two_week_2025_05_11/lgcp_metrics_first_day.csv


mean_ll_obs   mae_obs  rmse_obs  mean_ll_daily  \
method      week                                                       
LGCP (ours) may         -0.109853  0.067076  0.177882      -1.307222   
            november    -0.005215  0.005215  0.007830      -0.255534   

                      mae_daily  rmse_daily  daily_delta_corr  \
method      week                                                
LGCP (ours) may        0.038693    0.038693               NaN   
            november   0.255534    0.255534               NaN   

                      daily_direction_accuracy_moving  
method      week                                       
LGCP (ours) may                                   NaN  
            november                              NaN


Winning config: lr=0.01, temporal_kernel=week_month (['week', 'month'])
All 3 LGCP tables saved under /home/rdoz/PhD/AIPS/reports/paper/lgcp_two_week_2025_05_11
